# 01 --- Training: Tiers 1-5One arm per run. Checkpoints resume automatically, so a Colab disconnect costsminutes, not the whole run.- **Tier 1** smoke test- **Tier 2** Q1 architecture comparison- **Tier 3** Q2 data efficiency- **Tier 4** seeds and statistics- **Tier 5** ablations

## 1.0 --- Setup (run 00_verify first)

In [ ]:
# 0.1 --- environment!pip -q install "monai==1.4.0" einops nibabel scipy scikit-image!nvidia-smi --query-gpu=name,memory.total --format=csvfrom google.colab import drive; drive.mount('/content/drive')import os, sys, subprocessREPO = '/content/499A'if not os.path.exists(REPO):    !git clone https://github.com/ahnaf-csg/499A---3d-Tumor-Segmentation.git {REPO}else:    subprocess.run(['git','-C',REPO,'pull'])sys.path.insert(0, REPO)import torch, monaiprint('torch', torch.__version__, '| monai', monai.__version__, '| cuda', torch.cuda.is_available())assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU'cap = torch.cuda.get_device_capability(0)print(f'SM {cap[0]}.{cap[1]}  bf16={"yes" if cap[0]>=8 else "NO -> fp16 path (T4)"}')

In [ ]:
# stage data (fast path, tar already on Drive)DRIVE = '/content/drive/MyDrive/Colab Notebooks/499a'LOCAL = '/content/data'!mkdir -p {LOCAL}import osfor name in ['BraTS2021_Training_Data']:    if not os.path.exists(f'{LOCAL}/{name}'):        !cp "{DRIVE}/{name}.tar" /content/ && tar -xf /content/{name}.tar -C {LOCAL} && rm /content/{name}.tar!du -sh {LOCAL}/*from glioseg.config import Config, tier2_architectures, tier3_data_efficiency, tier4_seeds, tier5_ablationsfrom glioseg.train import train_arm, run_gridSPLIT = f'{DRIVE}/results/split_brats2021.json'OUT   = f'{DRIVE}/artifacts'BASE = Config(name='b21', dataset='brats2021', data_base=LOCAL,              model='segresnet', patch_size=(64,64,64), batch_size=2,              n_cases=300, epochs=30, val_every=5, lr=1e-4,              cache_dir='/content/cache', out_root=OUT, seed=42)print('run dir ->', BASE.run_dir.name)

## 1.1 --- Tier 1: smoke testTwo epochs. If this fails nothing below will work.**Then deliberately interrupt the next cell and re-run it** to provecheckpoint/resume works before you trust it with a paid 4-hour run.

In [ ]:
smoke = BASE.variant(name='smoke', epochs=2, val_every=1, n_cases=40)_ = train_arm(smoke, split_path=None)     # own split; does not touch the frozen one

## 1.2 --- Tier 2: Q1 architecture comparisonOnly `model` varies, so any difference is attributable to the architecture. SwinUNETR is the slow one — consider running it in its own A100 session.

In [ ]:
arch_cfgs = tier2_architectures(BASE, arms=('segresnet','unet3d','segformer3d'))arch_results = run_grid(arch_cfgs, split_path=SPLIT)import pandas as pdcols = ['model','params_M','model_size_MB','mean_dice_mean','WT_dice_mean',        'TC_dice_mean','ET_dice_mean','mean_epoch_time_s','error']df = pd.DataFrame(arch_results)df[[c for c in cols if c in df.columns]]

In [ ]:
# SwinUNETR separately -- switch to A100 first if units allowswin = BASE.variant(model='swinunetr', name='b21-arch')arch_results.append(train_arm(swin, split_path=SPLIT))

## 1.3 --- Tier 3: Q2 data efficiencyHow much data does the best arm need? This also retroactively justifies every subset used elsewhere — the training-set size *is* the experiment.

In [ ]:
ok = [r for r in arch_results if 'error' not in r]best = max(ok, key=lambda r: r.get('mean_dice_mean') or -1)print('best arm:', best['model'], best.get('mean_dice_mean'))de_cfgs = tier3_data_efficiency(BASE.variant(model=best['model']),                                sizes=(150, 300, 600, None))de_results = run_grid(de_cfgs, split_path=SPLIT)pd.DataFrame(de_results)[['n_cases','mean_dice_mean','ET_dice_mean','mean_epoch_time_s']]

## 1.4 --- Tier 4: seedsThree seeds on the primary arm only. Without this every claim rests on one run.

In [ ]:
seed_cfgs = tier4_seeds(BASE.variant(model=best['model'], name='b21-seed'))seed_results = run_grid(seed_cfgs, split_path=SPLIT)pd.DataFrame(seed_results)[['seed','mean_dice_mean','ET_dice_mean']]

## 1.5 --- Tier 5: ablationsOne factor at a time: loss, modality count, patch size. Post-processing is inference-only and handled in notebook 02.

In [ ]:
abl_cfgs = tier5_ablations(BASE.variant(model=best['model'], name='b21-abl'))abl_results = run_grid(abl_cfgs, split_path=SPLIT)pd.DataFrame(abl_results)[['name','loss','patch','n_modalities','mean_dice_mean','ET_dice_mean']]

## 1.6 --- Tier 7 (optional): Q3 pre-op to post-treatment transfer**The novelty.** Needs MU-Glioma-Post staged and a pre-op pretrained checkpoint.MONAI's `brats_mri_segmentation` bundle is SegResNet trained on BraTS 2018 — pre-operative, 4-in/3-out sigmoid, exactly our representation, so no head surgery.

In [ ]:
# stage MU + fetch pre-op weightsimport osfor name in ['MU-Glioma-Post']:    if not os.path.exists(f'{LOCAL}/{name}'):        !cp "{DRIVE}/{name}.tar" /content/ && tar -xf /content/{name}.tar -C {LOCAL} && rm /content/{name}.tarW = '/content/preop_segresnet.pt'if not os.path.exists(W):    !wget -q -O {W} https://huggingface.co/MONAI/brats_mri_segmentation/resolve/main/models/model.pt!ls -lh {W}from glioseg.config import tier7_transferfrom glioseg.verify import run_allrun_all(LOCAL, datasets=('mu_post',), sample=15,        log_path=f'{DRIVE}/results/verification_log.jsonl')   # verify BEFORE trainingMU_SPLIT = f'{DRIVE}/results/split_mu_post.json'tr_cfgs = tier7_transfer(BASE.variant(name='xfer', n_cases=None, epochs=25),                         pretrained_path=W, arms=('segresnet',))transfer_results = run_grid(tr_cfgs, split_path=MU_SPLIT)pd.DataFrame(transfer_results)[['name','pretrained','mean_dice_mean','ET_dice_mean']]